# TermNorm Pipeline Analysis

```
INITIALIZE -> S1 -> GROW/FILTER -> ANALYSIS -> [HUMAN]
```

Analyze the full TermNorm pipeline:
1. **Web Research** - Search + entity profile extraction
2. **Token Matching** - Fuzzy match against reference terms
3. **LLM Ranking** - Re-rank candidates with LLM

This notebook loads historical traces and analyzes pipeline performance.

## Cell 1: INITIALIZE - Load Dataset

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

# Add backend to path
sys.path.insert(0, '..')

# Load traces from experiments
EXPERIMENT = "1_production_historical"
TRACES_DIR = Path(f"../logs/experiments/{EXPERIMENT}/traces")

print(f"Loading traces from: {TRACES_DIR}")
print(f"Exists: {TRACES_DIR.exists()}")

In [ ]:
# Load all traces
traces = []

if TRACES_DIR.exists():
    for trace_dir in TRACES_DIR.iterdir():
        if trace_dir.is_dir():
            trace_info_file = trace_dir / "trace_info.yaml"
            if trace_info_file.exists():
                # Parse simple YAML
                trace_data = {"trace_id": trace_dir.name}
                with open(trace_info_file) as f:
                    for line in f:
                        if ":" in line:
                            key, value = line.strip().split(":", 1)
                            trace_data[key] = value.strip().strip('"')
                
                # Load tags
                tags_dir = trace_dir / "tags"
                if tags_dir.exists():
                    for tag_file in tags_dir.iterdir():
                        if tag_file.is_file():
                            trace_data[f"tag_{tag_file.name}"] = tag_file.read_text().strip()
                
                # Parse input/output from tags
                input_str = trace_data.get("tag_mlflow.traceInputs", "{}")
                output_str = trace_data.get("tag_mlflow.traceOutputs", "{}")
                try:
                    trace_data["input"] = json.loads(input_str)
                    trace_data["output"] = json.loads(output_str)
                except:
                    trace_data["input"] = {"raw": input_str}
                    trace_data["output"] = {"raw": output_str}
                
                traces.append(trace_data)

print(f"Loaded {len(traces)} traces")

In [ ]:
# Preview traces
if traces:
    preview = []
    for t in traces[:15]:
        preview.append({
            "trace_id": t["trace_id"][:12],
            "query": t.get("input", {}).get("query", "?")[:40],
            "target": t.get("output", {}).get("target", "?")[:50],
            "method": t.get("output", {}).get("method", "?"),
            "confidence": t.get("output", {}).get("confidence", "?"),
        })
    
    df = pd.DataFrame(preview)
    display(df)
else:
    print("No traces found")

## Cell 2: S1 (RUN) - Explore Pipeline Details

In [ ]:
# Load detailed spans for a specific trace
def load_trace_spans(trace_id):
    """Load spans (pipeline steps) for a trace."""
    trace_dir = TRACES_DIR / trace_id
    spans_file = trace_dir / "artifacts" / "traces.json"
    
    if spans_file.exists():
        data = json.loads(spans_file.read_text())
        return data.get("spans", [])
    return []

# Example: load first trace's spans
if traces:
    example_trace = traces[0]
    spans = load_trace_spans(example_trace["trace_id"])
    print(f"Trace: {example_trace['trace_id']}")
    print(f"Query: {example_trace.get('input', {}).get('query')}")
    print(f"\nPipeline steps ({len(spans)} spans):")
    for s in spans:
        print(f"  - {s.get('name', 'unnamed')}: {s.get('type', '?')}")

In [ ]:
# Analyze a specific span (e.g., token matching results)
def get_span_by_name(spans, name):
    for s in spans:
        if s.get("name") == name:
            return s
    return None

if spans:
    # Show token matching candidates
    token_span = get_span_by_name(spans, "token_matching")
    if token_span:
        print("TOKEN MATCHING RESULTS:")
        candidates = token_span.get("output", {}).get("candidates", [])
        for i, (term, score) in enumerate(candidates[:10]):
            print(f"  {i+1}. [{score:.3f}] {term[:60]}")
    
    # Show LLM ranking
    llm_span = get_span_by_name(spans, "llm_ranking")
    if llm_span:
        print("\nLLM RANKING RESULTS:")
        ranked = llm_span.get("output", {}).get("ranked_candidates", [])
        for i, c in enumerate(ranked[:5]):
            print(f"  {i+1}. [{c.get('relevance_score', 0):.2f}] {c.get('candidate', '?')[:60]}")

## Cell 3: GROW/FILTER - Analyze Patterns

In [ ]:
# Analyze methods used across all traces
method_counts = {}
confidence_by_method = {}

for t in traces:
    method = t.get("output", {}).get("method", "unknown")
    confidence = t.get("output", {}).get("confidence", 0)
    
    method_counts[method] = method_counts.get(method, 0) + 1
    if method not in confidence_by_method:
        confidence_by_method[method] = []
    if isinstance(confidence, (int, float)):
        confidence_by_method[method].append(confidence)

print("Methods used:")
for method, count in sorted(method_counts.items(), key=lambda x: -x[1]):
    avg_conf = sum(confidence_by_method.get(method, [0])) / max(len(confidence_by_method.get(method, [1])), 1)
    print(f"  {method}: {count} traces (avg confidence: {avg_conf:.2f})")

In [ ]:
# Find low-confidence predictions (potential failures)
low_confidence = []
for t in traces:
    conf = t.get("output", {}).get("confidence", 1)
    if isinstance(conf, (int, float)) and conf < 0.5:
        low_confidence.append(t)

print(f"\nLow confidence predictions (<0.5): {len(low_confidence)} / {len(traces)}")

if low_confidence:
    low_conf_df = pd.DataFrame([{
        "query": t.get("input", {}).get("query", "?")[:40],
        "target": t.get("output", {}).get("target", "?")[:40],
        "confidence": t.get("output", {}).get("confidence"),
        "method": t.get("output", {}).get("method"),
    } for t in low_confidence[:10]])
    display(low_conf_df)

## Cell 4: ANALYSIS - Pipeline Performance

In [ ]:
# Summary statistics
print("=" * 60)
print("PIPELINE ANALYSIS SUMMARY")
print("=" * 60)
print(f"Total traces: {len(traces)}")
print(f"Low confidence (<0.5): {len(low_confidence)} ({100*len(low_confidence)/max(len(traces),1):.1f}%)")
print("\nMethods breakdown:")
for method, count in sorted(method_counts.items(), key=lambda x: -x[1]):
    print(f"  {method}: {count} ({100*count/len(traces):.1f}%)")
print("=" * 60)

In [ ]:
# Confidence distribution
import matplotlib.pyplot as plt

confidences = [t.get("output", {}).get("confidence", 0) for t in traces 
               if isinstance(t.get("output", {}).get("confidence"), (int, float))]

if confidences:
    plt.figure(figsize=(10, 4))
    plt.hist(confidences, bins=20, edgecolor='black')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.title('Distribution of Confidence Scores')
    plt.axvline(x=0.5, color='r', linestyle='--', label='Threshold (0.5)')
    plt.legend()
    plt.show()

## Cell 5: [HUMAN] - Review & Experiment

In [ ]:
print("""
================================================================================
                           HUMAN REVIEW
================================================================================

Review the analysis above. Key questions:

1. Are low-confidence predictions actually wrong, or just uncertain?
2. Which pipeline step is the bottleneck (web search, token matching, LLM)?
3. Would adjusting the LLM ranking prompt help?

Experiments to try:
- [ ] Re-run low-confidence items with different prompt
- [ ] Compare with/without LLM re-ranking
- [ ] Test different web search parameters

================================================================================
""")

In [ ]:
# Quick experiment: Run pipeline on a single query
# (requires server to be running)

import httpx

async def run_single_query(query: str, skip_llm: bool = False):
    """Run the TermNorm pipeline on a single query."""
    async with httpx.AsyncClient(timeout=60.0) as client:
        # Initialize session with some reference terms
        # (In production, these come from the mapping file)
        resp = await client.post(
            "http://127.0.0.1:8000/sessions",
            json={"terms": ["example-term-1", "example-term-2"]}
        )
        print(f"Session: {resp.status_code}")
        
        # Run matching
        resp = await client.post(
            "http://127.0.0.1:8000/matches",
            json={"query": query, "skip_llm_ranking": skip_llm}
        )
        return resp.json()

# Uncomment to test:
# result = await run_single_query("steel coil")
# print(json.dumps(result, indent=2))